In [ ]:
from tabulate import tabulate
from astropy.coordinates import SkyCoord
from astropy.table import Table
import astropy.units as u
import numpy as np

In [ ]:
fits_path = '../output'

# Load asterisms
fits_file = f"{fits_path}/asterisms-GNAO-Optimal.fits"
#fits_file = f"{fits_path}/asterisms-GNAO-Nominal.fits"
#fits_file = f"{fits_path}/asterisms-GNAO-Limit.fits"
asterisms = Table.read(fits_file, format='fits')
print('Number of asterisms:', len(asterisms))

# Load targets
fits_file = f"{fits_path}/sample-targets.fits"
targets = Table.read(fits_file, format='fits')

print('Number of targets:', len(targets))

In [ ]:
def get_asterism_code(star1_mag, star2_mag, star3_mag):
    bri_limit = 15.0
    nom_limit = 17.0
    dim_limit = 18.5

    codes = []
    for i in range(len(star1_mag)):
        num_bright = 0
        num_nominal = 0
        num_dim = 0

        if star1_mag[i] != -1:
            if star1_mag[i] < bri_limit:
                num_bright += 1
            elif star1_mag[i] < nom_limit:
                num_nominal += 1
            elif star1_mag[i] < dim_limit:
                num_dim += 1

        if star2_mag[i] != -1:
            if star2_mag[i] < bri_limit:
                num_bright += 1
            elif star2_mag[i] < nom_limit:
                num_nominal += 1
            elif star2_mag[i] < dim_limit:
                num_dim += 1

        if star3_mag[i] != -1:
            if star3_mag[i] < bri_limit:
                num_bright += 1
            elif star3_mag[i] < nom_limit:
                num_nominal += 1
            elif star3_mag[i] < dim_limit:
                num_dim += 1

        codes.append(f"{num_bright}{num_nominal}{num_dim}")
    
    return np.array(codes)

In [ ]:
# Match targets to asterisms
asterism_catalog = SkyCoord(ra=asterisms['ra'], dec=asterisms['dec'], unit='deg', frame='icrs')
if isinstance(targets['ra'][0], str) and ':' in targets['ra'][0]:
    targets_catalog = SkyCoord(ra=targets['ra'], dec=targets['dec'], unit=(u.hourangle, u.deg), frame='icrs')
else:
    targets_catalog = SkyCoord(ra=targets['ra'], dec=targets['dec'], unit=(u.deg, u.deg), frame='icrs')
idx, sep, _ = targets_catalog.match_to_catalog_sky(asterism_catalog)
closest_asterism_id = asterisms['id'][idx]
closest_asterism_code = get_asterism_code(asterisms['star1_mag'][idx], asterisms['star2_mag'][idx], asterisms['star3_mag'][idx])

fov = 2*u.arcmin
target_filter = sep < fov/2

matches = targets[target_filter]
matches['asterism_id'] = closest_asterism_id[target_filter]
matches['asterism_code'] = closest_asterism_code[target_filter]
if 'field' in matches.colnames:
    matches.sort(['field', 'id'])
else:
    matches.sort(['id'])

print('Number of targets near an optimal asterism:', len(matches))
display(tabulate(matches, headers=matches.colnames, tablefmt='html', 
                 floatfmt=("", ".0f", ".5f", ".5f", ".3f", ".1f", ".1f", ".1f", ".0f")))